In [2]:
import os
from pathlib import Path
from torch_geometric.datasets import QM9

DATA_ROOT = Path.cwd().parent / "data" / "QM9" 

dataset = QM9(DATA_ROOT)

/opt/miniconda3/envs/equi-mac/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/miniconda3/envs/equi-mac/lib/python3.11/site-packages/torch_geometric/data/dataset.py:115: UserWarning: The `pre_transform` argument differs from the one used in the pre-processed version of this dataset. If you want to make use of another pre-processing technique, pass `force_reload=True` explicitly to reload the dataset.
  self._process()


SPHERICAL UTILS

In [3]:
import torch


def sphere_normalize(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    Normalize each position independently onto S^3.
    """
    return x / x.norm(dim=-1, keepdim=True).clamp_min(eps)


def probs_to_sphere(p: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    Map simplex probabilities to positive orthant of the sphere:
        phi(p) = sqrt(p)
    p: [N_atoms, 5], sum=1 along last dim
    Returns x in S^3_+ at each position: [N_atoms, 5]
    """
    p = p.clamp_min(eps)
    p = p / p.sum(dim=-1, keepdim=True).clamp_min(eps)
    x = torch.sqrt(p)
    return sphere_normalize(x, eps)


def sphere_to_probs(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    Inverse map of phi on S^3_+:
        p_i = x_i^2
    x: [N_atoms, 5]
    returns probs: [N_atoms, 5]
    """
    x = sphere_normalize(x, eps).clamp_min(0.0)
    p = x.pow(2)
    return p / p.sum(dim=-1, keepdim=True).clamp_min(eps)


def sphere_project_tangent(x: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """
    Project v onto tangent space at x, positionwise.
    x, v: [N_atoms, 5]
    """
    return v - (x * v).sum(dim=-1, keepdim=True) * x


def product_tangent_norm(v: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    Product metric norm:
        ||v||_g^2 = sum_l ||v_l||_2^2
    v: [N_atoms, 5]
    returns: [N_atoms, 1]
    """
    return torch.clamp(torch.norm(v, dim=-1, keepdim=True), min=1e-8)


def sphere_log(x: torch.Tensor, y: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    Positionwise sphere log map on S^3.

    x, y: [N_atoms, 5], each position normalized.
    returns: [N_atoms, 5], tangent at x
    """
    x = sphere_normalize(x, eps)
    y = sphere_normalize(y, eps)

    dot = (x * y).sum(dim=-1, keepdim=True).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
    theta = torch.acos(dot)  # [..., L, 1]

    u = y - dot * x
    u_norm = u.norm(dim=-1, keepdim=True)

    scale = theta / u_norm.clamp_min(eps)
    out = scale * u

    small = theta < 1e-5
    first_order = sphere_project_tangent(x, y - x)
    out = torch.where(small, first_order, out)
    return sphere_project_tangent(x, out)


def sphere_exp(x: torch.Tensor, v: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    Positionwise sphere exp map on S^3.

    x: [N_atoms, 5] normalized
    v: [N_atoms, 5] tangent at x
    returns: [N_atoms, 5] on sphere
    """
    x = sphere_normalize(x, eps)
    v = sphere_project_tangent(x, v)

    v_norm = v.norm(dim=-1, keepdim=True)
    direction = v / v_norm.clamp_min(eps)

    out = torch.cos(v_norm) * x + torch.sin(v_norm) * direction
    approx = sphere_normalize(x + v, eps)

    out = torch.where(v_norm < 1e-5, approx, out)
    return sphere_normalize(out, eps)


def geodesic_distance(
    x: torch.Tensor,  # [N_x, N_atoms, 5]
    y: torch.Tensor,  # [N_y, N_atoms, 5]
    eps: float = 1e-8,
) -> torch.Tensor:
    x = sphere_normalize(x, eps)
    y = sphere_normalize(y, eps)

    dot = (x * y).sum(dim=-1).clamp(-1.0 + 1e-7, 1.0 - 1e-7) 
    return torch.acos(dot)  # [N_mol, max_atoms]

EGNN

In [4]:
import torch
from torch import nn
from torch_geometric.nn import MessagePassing

class TypeGCN(MessagePassing):
    propagate_type = {"type_feat": torch.Tensor, "edge_attr": torch.Tensor}

    def __init__(
        self, hidden_nf: int, attention: bool = True, aggr_type: str = "sum"
    ):
        super().__init__(aggr=aggr_type)

        in_message_dim = hidden_nf * 2 + 2
        self.message_mlp = nn.Sequential(
            nn.Linear(in_message_dim, in_message_dim),
            nn.SiLU(),
            nn.Linear(in_message_dim, hidden_nf),
        )

        self.update_mlp = nn.Sequential(
            nn.Linear(hidden_nf * 2, hidden_nf),
            nn.SiLU(),
            nn.Linear(hidden_nf, hidden_nf),
        )

        self.attention = attention
        if self.attention:
            self.att_mlp = nn.Sequential(
                nn.Linear(hidden_nf, 1),
                nn.Sigmoid(),
            )

    def message(
        self,
        type_feat_i: torch.Tensor,
        type_feat_j: torch.Tensor,
        edge_attr: torch.Tensor
    ) -> torch.Tensor:
        input_tensor = torch.cat(
            [type_feat_i, type_feat_j, edge_attr],
            dim=-1,
        )
        out = self.message_mlp(input_tensor)

        if self.attention:
            att_val = self.att_mlp(out)
            return out * att_val

        return out

    def update(self, aggr_out: torch.Tensor, type_feat: torch.Tensor) -> torch.Tensor:
        out = torch.cat([aggr_out, type_feat], dim=-1)
        return type_feat + self.update_mlp(out)

    def forward(
        self, type_feat: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor
    ) -> torch.Tensor:
        return self.propagate(
            edge_index, type_feat=type_feat, edge_attr=edge_attr
        )


class PosGCN(MessagePassing):
    propagate_type = {
        "type_feat": torch.Tensor,
        "scaled_dir_vector": torch.Tensor,
        "edge_attr": torch.Tensor,
    }

    def __init__(
        self,
        hidden_nf: int,
        tanh_coord_updates: bool = True,
        coords_range: float = 15.0,
        aggr_type: str = "sum",
    ):
        super().__init__(aggr=aggr_type)

        in_message_dim = hidden_nf * 2 + 2

        layer = nn.Linear(hidden_nf, 1, bias=False)
        torch.nn.init.xavier_uniform_(layer.weight, gain=0.001)

        self.coord_mlp = nn.Sequential(
            nn.Linear(in_message_dim, hidden_nf),
            nn.SiLU(),
            nn.Linear(hidden_nf, hidden_nf),
            nn.SiLU(),
            layer,
        )

        self.tanh_coord_updates = tanh_coord_updates
        self.coords_range = coords_range

    def message(
        self,
        type_feat_i: torch.Tensor,
        type_feat_j: torch.Tensor,
        edge_attr: torch.Tensor,
        scaled_dir_vector: torch.Tensor,
    ) -> torch.Tensor:
        input_tensor = torch.cat([type_feat_i, type_feat_j, edge_attr], dim=-1)
        weight = self.coord_mlp(input_tensor)
        if self.tanh_coord_updates:
            weight = torch.tanh(weight) * self.coords_range
        return scaled_dir_vector * weight

    def forward(
        self,
        pos: torch.Tensor,
        type_feat: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr: torch.Tensor,
        scaled_dir_vector: torch.Tensor,
    ) -> torch.Tensor:
        delta = self.propagate(
            edge_index,
            type_feat=type_feat,
            edge_attr=edge_attr,
            scaled_dir_vector=scaled_dir_vector,
        )
        return pos + delta


class EquivariantBlock(nn.Module):
    def __init__(
        self,
        hidden_nf: int,
        n_layers: int = 1,
        attention: bool = True,
        tanh_coord_updates: bool = True,
        coords_range: float = 15.0,
        aggr_type: str = "sum",
    ):
        super().__init__()

        self.type_update = nn.ModuleList(
            [
                TypeGCN(hidden_nf, attention=attention, aggr_type=aggr_type)
                for _ in range(n_layers)
            ]
        )

        self.coord_update = PosGCN(
            hidden_nf,
            tanh_coord_updates=tanh_coord_updates,
            coords_range=coords_range,
            aggr_type=aggr_type,
        )

    def forward(
        self,
        type_feat: torch.Tensor,
        pos: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr: torch.Tensor,
        scaled_dir_vector: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        for type_gcn in self.type_update:
            type_feat = type_gcn(
                type_feat=type_feat,
                edge_index=edge_index,
                edge_attr=edge_attr
            )

        pos = self.coord_update(
            pos=pos,
            type_feat=type_feat,
            edge_index=edge_index,
            edge_attr=edge_attr,
            scaled_dir_vector=scaled_dir_vector,
        )

        return type_feat, pos

def compute_edge_properties(
    pos: torch.Tensor, edge_index: torch.Tensor, norm_constant: float = 1.0
) -> tuple[torch.Tensor, torch.Tensor]:
    src, dst = edge_index
    dist = pos[src] - pos[dst]
    squared_norm = dist.pow(2).sum(dim=-1)
    norm = squared_norm.sqrt()
    scaled_dir_vector = dist / (norm.unsqueeze(-1) + norm_constant)
    return squared_norm, scaled_dir_vector

class EGNN(nn.Module):
    def __init__(
        self,
        num_atom_types: int,
        num_blocks: int = 9,
        hidden_nf: int = 256,
        num_layers_per_block: int = 1,
        attention: bool = True,
        tanh_coord_updates: bool = True,
        coords_range: float = 15.0,
        aggr_type: str = "sum",
    ):
        super().__init__()
        self.type_embedding = nn.Linear(num_atom_types, hidden_nf)
        self.type_embedding_out = nn.Linear(hidden_nf, num_atom_types)
        self.blocks = nn.ModuleList(
            [
                EquivariantBlock(
                    hidden_nf,
                    n_layers=num_layers_per_block,
                    attention=attention,
                    tanh_coord_updates=tanh_coord_updates,
                    coords_range=coords_range,
                    aggr_type=aggr_type,
                )
                for _ in range(num_blocks)
            ]
        )

    def forward(
        self,
        pos_noise: torch.Tensor,
        type_noise: torch.Tensor,
        edge_index: torch.Tensor,
        return_change_in_pos: bool = False,
    ) -> tuple[torch.Tensor, torch.Tensor]:

        gen_feats = self.type_embedding(type_noise)
        gen_pos = pos_noise
        initial_squared_norm, _ = compute_edge_properties(gen_pos, edge_index)

        pos_list = []
        if return_change_in_pos:
            pos_list.append(gen_pos.clone())

        for block in self.blocks:
            squared_norm, scaled_dir_vector = compute_edge_properties(gen_pos, edge_index)
            edge_attr = torch.cat(
                [initial_squared_norm.unsqueeze(-1), squared_norm.unsqueeze(-1)],
                dim=-1,
            )
            gen_feats, gen_pos = block(
                type_feat=gen_feats,
                pos=gen_pos,
                edge_index=edge_index,
                scaled_dir_vector=scaled_dir_vector,
                edge_attr=edge_attr,
            )
            if return_change_in_pos:
                pos_list.append(gen_pos.clone())

        gen_types = self.type_embedding_out(gen_feats)

        if return_change_in_pos:
            return gen_pos, gen_types, pos_list
        return gen_pos, gen_types

In [5]:
import matplotlib.pyplot as plt


def plot_training_history(history):
    if not history:
        return

    steps = [row["step"] for row in history]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(steps, [row["loss"] for row in history], label="loss")
    axes[0].plot(steps, [row["position_loss"] for row in history], label="position")
    axes[0].plot(steps, [row["type_loss"] for row in history], label="type")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    diag_rows = [row for row in history if "diag_rmse" in row]
    if diag_rows:
        axes[1].plot([row["step"] for row in diag_rows], [row["diag_rmse"] for row in diag_rows], label="diag RMSE")
        axes[1].plot([row["step"] for row in diag_rows], [row["nearest_real_acc"] for row in diag_rows], label="nearest-real acc")
        axes[1].plot([row["step"] for row in diag_rows], [row["type_acc"] for row in diag_rows], label="type acc")
    axes[1].set_xlabel("Step")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


In [6]:
# Adding Kabsch algorithm and Hungarian method
from torch_linear_assignment import batch_linear_assignment

@torch.no_grad()
def _kabsch_rotations(gen_pos, real_pos):
    """
    Supports:
        gen_pos:  [N_gen, N_atoms, 3]
        gen_pos:  [N_gen, N_real, N_atoms, 3]
        real_pos: [N_real, N_atoms, 3]

    Returns:
        R: [N_gen, N_real, 3, 3]

    R[g, r] aligns gen_pos[g] or gen_pos[g, r] to real_pos[r]
    when rotations are applied to row vectors as gen_pos @ R.
    """
    if real_pos.ndim != 3:
        raise ValueError(f"real_pos must be [N_real, N_atoms, 3], got {real_pos.shape}")

    if gen_pos.ndim == 3:
        if gen_pos.shape[1:] != real_pos.shape[1:]:
            raise ValueError(
                f"Shape mismatch: gen_pos {gen_pos.shape}, real_pos {real_pos.shape}"
            )

        gen_c = gen_pos - gen_pos.mean(dim=1, keepdim=True)
        real_c = real_pos - real_pos.mean(dim=1, keepdim=True)

        # H[g, r] = gen_c[g].T @ real_c[r]
        H = torch.einsum("gni,rnj->grij", gen_c, real_c)

    elif gen_pos.ndim == 4:
        if gen_pos.shape[1] != real_pos.shape[0]:
            raise ValueError(
                f"Pairwise gen_pos has N_real={gen_pos.shape[1]}, "
                f"but real_pos has N_real={real_pos.shape[0]}"
            )
        if gen_pos.shape[2:] != real_pos.shape[1:]:
            raise ValueError(
                f"Shape mismatch: gen_pos {gen_pos.shape}, real_pos {real_pos.shape}"
            )

        gen_c = gen_pos - gen_pos.mean(dim=2, keepdim=True)
        real_c = real_pos - real_pos.mean(dim=1, keepdim=True)

        # H[g, r] = gen_c[g, r].T @ real_c[r]
        H = torch.einsum("grni,rnj->grij", gen_c, real_c)

    else:
        raise ValueError(f"gen_pos must be 3D or 4D, got {gen_pos.shape}")

    H_flat = H.reshape(-1, 3, 3)
    U, S, Vh = torch.linalg.svd(H_flat)

    R = U @ Vh
    det = torch.det(R)
    mask = det < 0

    if mask.any():
        U = U.clone()
        U[mask, :, -1] *= -1
        R = U @ Vh

    return R.reshape(H.shape[0], H.shape[1], 3, 3)


def _hungarian_method_batched(
    gen_types,
    real_types,
    sigma,
    eps,
    gen_pos=None,
    real_pos=None,
    type_weight=1.0,
    position_weight=1.0,
    position_sigma=1.0,
):
    cost_matrix = _build_cost_matrix(
        gen=gen_types,
        real=real_types,
        sigma=sigma,
        eps=eps,
        gen_pos=gen_pos,
        real_pos=real_pos,
        type_weight=type_weight,
        position_weight=position_weight,
        position_sigma=position_sigma,
    )

    N_gen = cost_matrix.shape[0]
    N_real = cost_matrix.shape[1]
    N_atoms = cost_matrix.shape[2]

    cost_flat = cost_matrix.reshape(N_gen * N_real, N_atoms, N_atoms).contiguous()

    with torch.no_grad():
        assignment_flat = batch_linear_assignment(cost_flat)

    assignment = assignment_flat.reshape(N_gen, N_real, N_atoms)

    return assignment


def _build_type_cost_matrix(gen, real, sigma, eps):
    """
    Supports:
        gen:  [N_gen, N_atoms, D]
        gen:  [N_gen, N_real, N_atoms, D]
        real: [N_real, N_atoms, D]

    returns:
        cost_matrix: [N_gen, N_real, N_atoms_real, N_atoms_gen]

    cost_matrix[g, r, j_real, i_gen] =
        type cost of assigning generated atom i to real atom j
    """
    assert real.ndim == 3

    gen = sphere_normalize(gen, eps)
    real = sphere_normalize(real, eps)

    if gen.ndim == 3:
        gen_expanded = gen[:, None, None, :, :]
        real_expanded = real[None, :, :, None, :]

    elif gen.ndim == 4:
        gen_expanded = gen[:, :, None, :, :]
        real_expanded = real[None, :, :, None, :]

    else:
        raise ValueError(f"Expected gen to be 3D or 4D, got shape {gen.shape}")

    dot = (gen_expanded * real_expanded).sum(dim=-1)
    dot = dot.clamp(-1.0 + 1e-7, 1.0 - 1e-7)

    dist = torch.acos(dot)

    return dist.pow(2) / (2 * sigma**2)


def _build_position_cost_matrix(gen_pos, real_pos, position_sigma=1.0):
    """
    Supports:
        gen_pos:  [N_gen, N_atoms, 3]
        gen_pos:  [N_gen, N_real, N_atoms, 3]
        real_pos: [N_real, N_atoms, 3]

    returns:
        cost_matrix: [N_gen, N_real, N_atoms_real, N_atoms_gen]
    """
    if real_pos.ndim != 3:
        raise ValueError(f"real_pos must be [N_real, N_atoms, 3], got {real_pos.shape}")

    if gen_pos.ndim == 3:
        gen_expanded = gen_pos[:, None, None, :, :]
        real_expanded = real_pos[None, :, :, None, :]

    elif gen_pos.ndim == 4:
        if gen_pos.shape[1] != real_pos.shape[0]:
            raise ValueError(
                f"Pairwise gen_pos has N_real={gen_pos.shape[1]}, "
                f"but real_pos has N_real={real_pos.shape[0]}"
            )
        gen_expanded = gen_pos[:, :, None, :, :]
        real_expanded = real_pos[None, :, :, None, :]

    else:
        raise ValueError(f"Expected gen_pos to be 3D or 4D, got shape {gen_pos.shape}")

    sq_dist = (gen_expanded - real_expanded).pow(2).sum(dim=-1)
    return sq_dist / (2 * position_sigma**2)


def _build_cost_matrix(
    gen,
    real,
    sigma,
    eps,
    gen_pos=None,
    real_pos=None,
    type_weight=1.0,
    position_weight=1.0,
    position_sigma=1.0,
):
    type_cost = _build_type_cost_matrix(gen, real, sigma, eps)
    cost_matrix = type_weight * type_cost

    if position_weight != 0.0:
        if gen_pos is None or real_pos is None:
            raise ValueError("gen_pos and real_pos are required when position_weight != 0")
        position_cost = _build_position_cost_matrix(
            gen_pos,
            real_pos,
            position_sigma=position_sigma,
        )
        cost_matrix = cost_matrix + position_weight * position_cost

    return cost_matrix


def _to_pairwise(gen, n_real):
    if gen.ndim == 3:
        return gen[:, None, :, :].expand(-1, n_real, -1, -1)
    if gen.ndim == 4:
        return gen
    raise ValueError(f"Expected 3D or 4D tensor, got {gen.shape}")


def _pairwise_position_rmse(gen_pos_pairwise, real_pos):
    """
    gen_pos_pairwise: [N_gen, N_real, N_atoms, 3]
    real_pos:         [N_real, N_atoms, 3]

    returns:
        rmse: [N_gen, N_real]
    """
    diff = gen_pos_pairwise - real_pos[None, :, :, :]
    return diff.pow(2).sum(dim=-1).mean(dim=-1).sqrt()


def permute_generated_to_real_order(gen, assignment):
    """
    Supports:
        gen: [N_gen, N_atoms, D]
        gen: [N_gen, N_real, N_atoms, D]

    assignment: [N_gen, N_real, N_atoms]
        assignment[g, r, j_real] = i_gen

    returns:
        gen_perm: [N_gen, N_real, N_atoms, D]
    """
    D = gen.shape[-1]

    if gen.ndim == 3:
        gen_pairwise = gen[:, None, :, :].expand(-1, assignment.shape[1], -1, -1)

    elif gen.ndim == 4:
        gen_pairwise = gen

    else:
        raise ValueError(f"Expected gen to be 3D or 4D, got shape {gen.shape}")

    idx = assignment[..., None].expand(-1, -1, -1, D)

    gen_perm = gen_pairwise.gather(dim=2, index=idx)

    return gen_perm


def apply_pairwise_rotation(gen_pos, R):
    """
    Supports:
        gen_pos: [N_gen, N_atoms, 3]
        gen_pos: [N_gen, N_real, N_atoms, 3]

    R: [N_gen, N_real, 3, 3]

    Returns:
        rotated: [N_gen, N_real, N_atoms, 3]
    """
    if gen_pos.ndim == 3:
        gen_pairwise = gen_pos[:, None, :, :].expand(-1, R.shape[1], -1, -1)
    elif gen_pos.ndim == 4:
        gen_pairwise = gen_pos
    else:
        raise ValueError(f"gen_pos must be 3D or 4D, got {gen_pos.shape}")

    return gen_pairwise @ R


def unpermute_real_order_to_gen_order(x_perm, assignment):
    """
    x_perm: [N_gen, N_real, N_atoms, D]
        tensor currently ordered by real atom index

    assignment: [N_gen, N_real, N_atoms]
        assignment[g, r, j_real] = i_gen

    returns:
        x: [N_gen, N_real, N_atoms, D]
        tensor ordered by original generated atom index
    """
    x = torch.empty_like(x_perm)

    idx = assignment[..., None].expand_as(x_perm)

    x.scatter_(dim=2, index=idx, src=x_perm)

    return x


def find_rotation_and_permutation(
    gen_pos,
    real_pos,
    gen_types,
    real_types,
    sigma,
    eps,
    max_iter,
    pos_tol=1e-1,
    min_iter=1,
    type_weight=1.0,
    position_weight=1.0,
    position_sigma=1.0,
    position_warmup_steps=1,
):
    """
    Alternates Hungarian assignment and Kabsch alignment.

    By default this uses the geometry_w1 setting. Pass
    position_weight=0.0 explicitly to run the type-only baseline.
    position_warmup_steps=1 means the first assignment is type-only, giving
    Kabsch one chance to find an initial rotation before position costs matter.
    """
    g_types = gen_types.clone().detach()
    g_pos = gen_pos.clone().detach()

    N_gen = gen_pos.shape[0]
    N_real = real_pos.shape[0]
    N_atoms = gen_pos.shape[1]

    active = torch.ones(N_gen, N_real, device=gen_pos.device, dtype=torch.bool)

    total_assignment = torch.arange(
        N_atoms,
        device=gen_pos.device,
    ).view(1, 1, N_atoms).expand(N_gen, N_real, N_atoms).clone()

    total_R = torch.eye(
        3,
        device=gen_pos.device,
        dtype=gen_pos.dtype,
    ).view(1, 1, 3, 3).expand(N_gen, N_real, 3, 3).clone()

    for step in range(max_iter):
        old_g_pos = _to_pairwise(g_pos, N_real)
        old_g_types = _to_pairwise(g_types, N_real)
        step_position_weight = 0.0 if step < position_warmup_steps else position_weight

        step_assignment = _hungarian_method_batched(
            g_types,
            real_types,
            sigma=sigma,
            eps=eps,
            gen_pos=g_pos,
            real_pos=real_pos,
            type_weight=type_weight,
            position_weight=step_position_weight,
            position_sigma=position_sigma,
        )

        cand_g_pos = permute_generated_to_real_order(g_pos, step_assignment)
        cand_g_types = permute_generated_to_real_order(g_types, step_assignment)

        step_R = _kabsch_rotations(cand_g_pos, real_pos)
        cand_g_pos = apply_pairwise_rotation(cand_g_pos, step_R)

        cand_total_assignment = total_assignment.gather(
            dim=2,
            index=step_assignment,
        )

        cand_total_R = total_R @ step_R

        pair_mask = active[..., None, None]
        assign_mask = active[..., None]

        g_pos = torch.where(pair_mask, cand_g_pos, old_g_pos)
        g_types = torch.where(pair_mask, cand_g_types, old_g_types)

        total_assignment = torch.where(
            assign_mask,
            cand_total_assignment,
            total_assignment,
        )

        total_R = torch.where(
            pair_mask,
            cand_total_R,
            total_R,
        )

        rmse = _pairwise_position_rmse(g_pos, real_pos)
        done = rmse <= pos_tol

        if step + 1 < min_iter:
            active = torch.ones_like(done)
        else:
            active = ~done

        if not active.any():
            break

    return total_assignment, total_R, g_pos, g_types


In [7]:
def compute_aligning_drift_loss(
    gen_pos: torch.Tensor,
    real_pos: torch.Tensor,
    gen_types_sphere: torch.Tensor,
    real_types: torch.Tensor,
    posit_sigma: float = 1.0,
    types_sigma: float = 1.0,
    eps: float = 1e-8,
    scale_loss: float = 1.0,
    alignment_kwargs: dict = None,
) -> tuple[torch.Tensor, dict[str, float]]:
    """
    Drifting field loss directly on 3D molecules.

    Alignment is used only as an internal coordinate system for computing
    pairwise distances and drift directions. The resulting drift vectors are
    then unrotated and unpermuted back to the original generated molecule frame
    before constructing targets. In other words: compute drift aligned, apply
    drift unaligned.
    """
    real_types = real_types.float()

    N_gen = gen_pos.shape[0]
    N_real = real_pos.shape[0]
    alignment_kwargs = alignment_kwargs or {}

    permutation_pos, R_pos, aligned_posit_pos, aligned_types_pos = find_rotation_and_permutation(
        gen_pos,
        real_pos,
        gen_types_sphere,
        real_types,
        posit_sigma,
        eps,
        max_iter=10,
        **alignment_kwargs,
    )
    permutation_neg, R_neg, aligned_posit_neg, aligned_types_neg = find_rotation_and_permutation(
        gen_pos,
        gen_pos,
        gen_types_sphere,
        gen_types_sphere,
        posit_sigma,
        eps,
        max_iter=10,
        **alignment_kwargs,
    )

    # Distances of shape [N_gen, N_real/N_gen] and Differences of shape [N_gen, N_real/N_gen, N_atoms, 3/5]
    posit_dist_pos, posit_diff_pos = _pairwise_geodesic_distance_and_log(aligned_posit_pos, real_pos, "euclidean", eps)
    posit_dist_neg, posit_diff_neg = _pairwise_geodesic_distance_and_log(aligned_posit_neg, gen_pos, "euclidean", eps)

    types_dist_pos, types_diff_pos = _pairwise_geodesic_distance_and_log(aligned_types_pos, real_types, "spherical", eps)
    types_dist_neg, types_diff_neg = _pairwise_geodesic_distance_and_log(aligned_types_neg, gen_types_sphere, "spherical", eps)

    # PERHAPS DIVIDE DIST BY SQRT N_ATOMS

    # Ignore self if y_neg is x
    eye = torch.eye(N_gen, device=gen_pos.device, dtype=torch.bool)
    posit_dist_neg = posit_dist_neg.masked_fill(eye, 1e6)
    types_dist_neg = types_dist_neg.masked_fill(eye, 1e6)
    
    V_posit = _compute_V_at_sigma(
        posit_dist_pos,
        posit_dist_neg,
        posit_diff_pos,
        posit_diff_neg,
        permutation_pos,
        R_pos,
        permutation_neg,
        R_neg,
        posit_sigma,
        eps,
    )
    V_types = _compute_V_at_sigma(
        types_dist_pos,
        types_dist_neg,
        types_diff_pos,
        types_diff_neg,
        permutation_pos,
        None,
        permutation_neg,
        None,
        types_sigma,
        eps,
        euclidean=False,
    )

    V_types = sphere_project_tangent(gen_types_sphere, V_types)
    # PERHAPS HERE ADD SCALING OF DRIFTING FIELD

    # Calculate targets for each 
    target_posit = (gen_pos + V_posit).detach()
    target_types = sphere_exp(gen_types_sphere, V_types, eps).detach()

    # Calculate distances per molecule on each manifold
    n_atoms = gen_pos.shape[-2]
    molecule_position_dist = ((gen_pos - target_posit) ** 2).sum(dim=-1).sum(dim=-1) / n_atoms
    molecule_types_dist = (geodesic_distance(gen_types_sphere, target_types, eps) ** 2).sum(dim=-1) / n_atoms

    loss = (molecule_position_dist + scale_loss * molecule_types_dist).mean()
    
    return loss, molecule_position_dist, molecule_types_dist


def _pairwise_geodesic_distance_and_log(
        x: torch.Tensor,
        y: torch.Tensor,
        manifold: str = "euclidean",
        eps: float = 1e-8,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Calculates the pairwise distance between atoms for the molecule attributes depending on 
    the specified geodesic distance. Assumed that molecules are aligned.
    Args:
        x: [N_x, N_y, N_atoms, z]
        y: [N_y, N_atoms, z]
    """

    if manifold == "euclidean":
        diff = x - y[None, :, :, :]                              # shape: (N_x, N_y, N_atoms, z)
        sq_dist_per_atom = (diff ** 2).sum(dim=-1)                              # shape: (N_x, N_y, N_atoms)

        sq_distances = sq_dist_per_atom.mean(dim=-1)  # shape: (N_x, N_y)
    elif manifold == "spherical":
        x = sphere_normalize(x, eps)
        y = sphere_normalize(y.unsqueeze(0), eps)

        dot = (x * y).sum(dim=-1).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
        theta = torch.acos(dot)  # [N_x, N_y, N_atoms]

        sq_distances = theta.pow(2).mean(dim=-1).clamp_min(eps)  # [N_x, N_y]

        u = y - dot.unsqueeze(-1) * x  # [N_x, N_y, N_atoms, z]
        u_norm = u.norm(dim=-1, keepdim=True)

        scale = theta.unsqueeze(-1) / u_norm.clamp_min(eps)
        out = scale * u

        small = theta.unsqueeze(-1) < 1e-5
        first_order = sphere_project_tangent(x, y - x)

        out = torch.where(small, first_order, out)
        diff = sphere_project_tangent(x, out)  # shape: (N_x, N_y, N_atoms, z)
    else:
        raise ValueError("Undefined manifold.")
    return sq_distances, diff


def _unalign_pairwise_gradient(grad, permutation, R=None):
    if R is not None:
        grad = grad @ R.transpose(-2, -1)
    return unpermute_real_order_to_gen_order(grad, permutation)


def _compute_V_at_sigma(
    dist_pos,
    dist_neg,
    diff_pos,
    diff_neg,
    permutation_pos,
    R_pos,
    permutation_neg,
    R_neg,
    sigma,
    eps=1e-8,
    euclidean=True,
):
    # Calculate kernels
    kernel_pos = torch.exp(-dist_pos / (2 * sigma**2))   # shape [N_gen, N_real]
    kernel_neg = torch.exp(-dist_neg / (2 * sigma**2))   # shape [N_gen, N_real]

    # Calculate gradient of kernels towards the aligned generated input.
    grad_kernel_pos = (diff_pos * kernel_pos.unsqueeze(-1).unsqueeze(-1)) / (sigma**2)
    grad_kernel_neg = (diff_neg * kernel_neg.unsqueeze(-1).unsqueeze(-1)) / (sigma**2)

    if euclidean:
        grad_kernel_pos = -grad_kernel_pos
        grad_kernel_neg = -grad_kernel_neg

    grad_kernel_pos = _unalign_pairwise_gradient(grad_kernel_pos, permutation_pos, R_pos)
    grad_kernel_neg = _unalign_pairwise_gradient(grad_kernel_neg, permutation_neg, R_neg)

    V_pos = grad_kernel_pos.sum(dim=1) / kernel_pos.sum(dim=1).clamp_min(eps).unsqueeze(-1).unsqueeze(-1)
    V_neg = grad_kernel_neg.sum(dim=1) / kernel_neg.sum(dim=1).clamp_min(eps).unsqueeze(-1).unsqueeze(-1)

    V_at_sigma = V_pos - V_neg
    return V_at_sigma


# Geometry_w1 comparison baseline

Type-only is kept only as a diagnostic baseline. The alignment and drift defaults are geometry_w1.


In [8]:
GEOMETRY_W1_ALIGNMENT_KWARGS = dict(
    position_weight=1.0,
    position_sigma=1.0,
    position_warmup_steps=1,
)

TYPE_ONLY_ALIGNMENT_KWARGS = dict(
    position_weight=0.0,
    position_sigma=1.0,
    position_warmup_steps=1,
)


def _random_proper_rotation(device, dtype):
    A = torch.randn(3, 3, device=device, dtype=dtype)
    Q, _ = torch.linalg.qr(A)
    if torch.det(Q) < 0:
        Q[:, -1] *= -1
    return Q


def _fixed_size_indices(size, max_count, max_scan=50000):
    indices = []
    for idx in range(min(max_scan, len(dataset))):
        if int(dataset[idx].num_nodes) == size:
            indices.append(idx)
        if len(indices) >= max_count:
            break
    return indices


def _make_alignment_comparison_batch(size=8, batch_size=32, seed=0, coord_noise_std=0.05, type_noise_level=0.10):
    torch.manual_seed(seed)
    mol_indices = _fixed_size_indices(size=size, max_count=batch_size)
    if len(mol_indices) < batch_size:
        print(f"Only found {len(mol_indices)} molecules with size={size}; using all of them")

    real_pos_list = []
    real_types_list = []
    gen_pos_list = []
    gen_types_list = []
    perms = []

    for idx in mol_indices:
        mol = dataset[idx]
        mol_pos = mol.pos.float()
        mol_types = mol.real_atom_types.float()
        n_atoms = int(mol.num_nodes)

        mol_pos = mol_pos - mol_pos.mean(dim=0, keepdim=True)
        perm = torch.randperm(n_atoms, device=mol_pos.device)
        R_true = _random_proper_rotation(mol_pos.device, mol_pos.dtype)

        gen_pos = mol_pos[perm] @ R_true
        if coord_noise_std > 0:
            gen_pos = gen_pos + coord_noise_std * torch.randn_like(gen_pos)
        gen_pos = gen_pos - gen_pos.mean(dim=0, keepdim=True)

        gen_types = mol_types[perm]
        if type_noise_level > 0:
            noise = torch.rand_like(gen_types)
            noise = noise / noise.sum(dim=-1, keepdim=True).clamp_min(1e-8)
            gen_types = (1.0 - type_noise_level) * gen_types + type_noise_level * noise

        real_pos_list.append(mol_pos)
        real_types_list.append(mol_types)
        gen_pos_list.append(gen_pos)
        gen_types_list.append(gen_types)
        perms.append(perm)

    return {
        "gen_pos": torch.stack(gen_pos_list, dim=0),
        "gen_types": torch.stack(gen_types_list, dim=0),
        "real_pos": torch.stack(real_pos_list, dim=0),
        "real_types": torch.stack(real_types_list, dim=0),
        "perms": torch.stack(perms, dim=0),
        "indices": mol_indices,
    }


def _alignment_summary(name, batch, alignment_kwargs):
    assignment, R, aligned_pos, aligned_types = find_rotation_and_permutation(
        batch["gen_pos"],
        batch["real_pos"],
        batch["gen_types"],
        batch["real_types"],
        sigma=1.0,
        eps=1e-8,
        max_iter=100,
        **alignment_kwargs,
    )

    n = batch["gen_pos"].shape[0]
    arange = torch.arange(n, device=batch["gen_pos"].device)
    pair_rmse = (aligned_pos - batch["real_pos"][None, :, :, :]).pow(2).sum(dim=-1).mean(dim=-1).sqrt()
    diag_rmse = pair_rmse[arange, arange]
    nearest_real = pair_rmse.argmin(dim=1)
    expected_assignment = torch.argsort(batch["perms"], dim=1)
    assignment_acc = (assignment[arange, arange] == expected_assignment).float().mean(dim=-1)
    type_acc = (
        aligned_types[arange, arange].argmax(dim=-1) == batch["real_types"].argmax(dim=-1)
    ).float().mean(dim=-1)

    return {
        "method": name,
        "mean_diag_rmse": diag_rmse.mean().item(),
        "median_diag_rmse": diag_rmse.median().item(),
        "max_diag_rmse": diag_rmse.max().item(),
        "mean_type_acc": type_acc.mean().item(),
        "mean_assignment_acc": assignment_acc.mean().item(),
        "nearest_real_acc": (nearest_real == arange).float().mean().item(),
    }


def run_geometry_w1_alignment_comparison(size=8, batch_size=32, seeds=(0, 1, 2), coord_noise_std=0.05, type_noise_level=0.10):
    rows = []
    for seed in seeds:
        batch = _make_alignment_comparison_batch(
            size=size,
            batch_size=batch_size,
            seed=seed,
            coord_noise_std=coord_noise_std,
            type_noise_level=type_noise_level,
        )
        rows.append(_alignment_summary("type_only", batch, TYPE_ONLY_ALIGNMENT_KWARGS))
        rows.append(_alignment_summary("geometry_w1", batch, GEOMETRY_W1_ALIGNMENT_KWARGS))

    for method in ["type_only", "geometry_w1"]:
        method_rows = [row for row in rows if row["method"] == method]
        print(
            f"{method:>11} | "
            f"diag RMSE mean={sum(row['mean_diag_rmse'] for row in method_rows) / len(method_rows):.6f} | "
            f"type acc={sum(row['mean_type_acc'] for row in method_rows) / len(method_rows):.4f} | "
            f"assignment acc={sum(row['mean_assignment_acc'] for row in method_rows) / len(method_rows):.4f} | "
            f"nearest-real acc={sum(row['nearest_real_acc'] for row in method_rows) / len(method_rows):.4f}"
        )

    return rows


comparison_rows = run_geometry_w1_alignment_comparison()


  type_only | diag RMSE mean=1.601655 | type acc=1.0000 | assignment acc=0.4206 | nearest-real acc=0.1354
geometry_w1 | diag RMSE mean=0.425071 | type acc=0.9427 | assignment acc=0.6003 | nearest-real acc=0.6771


# Geometry-aware drifting model training test

This trains an EGNN on a larger fixed-size real molecule batch using the `geometry_w1` alignment setting. The inputs are regenerated each step with random rotations, atom permutations, coordinate noise, and type noise.

In [38]:
DRIFT_TRAIN_CONFIG = {
    "size": 6,
    "batch_size": 10,
    "max_scan": 50000,
    "seed": 123,
    "device": "cpu",
    "steps": 3000,
    "log_every": 10,
    "lr": 1e-5,
    "lr_schedule": [
        (800, 0.25),
        (1500, 0.05),
    ],
    "weight_decay": 1e-5,
    "grad_clip": 1.0,
    "checkpoint_every": 10,
    "checkpoint_path": "geometry_w1_best_checkpoint.pt",
    "hidden_nf": 128,
    "num_blocks": 4,
    "num_layers_per_block": 1,
    "coords_range": 5.0,
    "posit_sigma": 1.5,
    "types_sigma": 1.5,
    "scale_loss": 1.0,
    "lambda_drift": 1.0,
    "lambda_clash": 0.02,
    "lambda_valence_excess": 0.02,
    "lambda_hydrogen_valence": 0.05,
    "chemistry_checkpoint_every": 200,
    "clash_threshold": 0.7,
    "bond_threshold_scale": 1.25,
    "bond_temperature": 0.4,
    "alignment_kwargs": dict(
        **GEOMETRY_W1_ALIGNMENT_KWARGS,
    ),
}


In [39]:
def _select_real_molecule_batch(size, batch_size, max_scan, device):
    real_pos_list = []
    real_types_list = []
    selected_indices = []

    for idx in range(min(max_scan, len(dataset))):
        mol = dataset[idx]
        if int(mol.num_nodes) != size:
            continue

        mol_pos = mol.pos.float()
        mol_pos = mol_pos - mol_pos.mean(dim=0, keepdim=True)
        mol_types = mol.real_atom_types.float()

        real_pos_list.append(mol_pos)
        real_types_list.append(mol_types)
        selected_indices.append(idx)

        if len(selected_indices) >= batch_size:
            break

    if not selected_indices:
        raise ValueError(f"No molecules with size={size} found in first {max_scan} dataset entries")

    if len(selected_indices) < batch_size:
        print(f"Only found {len(selected_indices)} molecules with size={size}; using all of them")

    return (
        torch.stack(real_pos_list, dim=0).to(device),
        torch.stack(real_types_list, dim=0).to(device),
        selected_indices,
    )


def _make_complete_edge_index_fixed_batch(batch_size, n_atoms, device):
    nodes = torch.arange(n_atoms, device=device)
    src = nodes.repeat_interleave(n_atoms)
    dst = nodes.repeat(n_atoms)
    mask = src != dst
    base_edges = torch.stack([src[mask], dst[mask]], dim=0)

    offsets = (torch.arange(batch_size, device=device) * n_atoms).view(batch_size, 1, 1)
    edges = base_edges.view(1, 2, -1) + offsets
    return edges.permute(1, 0, 2).reshape(2, -1).contiguous()


def _soft_pair_threshold(pred_type_probs, thresholds):
    thresholds = thresholds.to(pred_type_probs.device, dtype=pred_type_probs.dtype)
    return torch.einsum("bik,kl,bjl->bij", pred_type_probs, thresholds, pred_type_probs)


def _soft_pair_compatibility(pred_type_probs, thresholds):
    mask = (thresholds > 0).to(pred_type_probs.device, dtype=pred_type_probs.dtype)
    return torch.einsum("bik,kl,bjl->bij", pred_type_probs, mask, pred_type_probs)


BOND_COVALENT_RADII = torch.tensor([0.31, 0.76, 0.71, 0.66, 0.57])
ATOM_TYPICAL_VALENCE = torch.tensor([1.0, 4.0, 3.0, 2.0, 1.0])

# Soft EDM/QM9 bond-order thresholds in Angstroms, matching mol_utils.bonds margins.
BOND_ORDER_1_THRESHOLDS = torch.tensor([
    [0.84, 1.19, 1.11, 1.06, 1.02],
    [1.19, 1.64, 1.57, 1.53, 1.45],
    [1.11, 1.57, 1.55, 1.50, 1.46],
    [1.06, 1.53, 1.50, 1.58, 1.52],
    [1.02, 1.45, 1.46, 1.52, 1.52],
])
BOND_ORDER_2_THRESHOLDS = torch.tensor([
    [0.00, 0.00, 0.00, 0.00, 0.00],
    [0.00, 1.39, 1.34, 1.25, 0.00],
    [0.00, 1.34, 1.30, 1.26, 0.00],
    [0.00, 1.25, 1.26, 1.26, 0.00],
    [0.00, 0.00, 0.00, 0.00, 0.00],
])
BOND_ORDER_3_THRESHOLDS = torch.tensor([
    [0.00, 0.00, 0.00, 0.00, 0.00],
    [0.00, 1.23, 1.19, 1.16, 0.00],
    [0.00, 1.19, 1.13, 0.00, 0.00],
    [0.00, 1.16, 0.00, 0.00, 0.00],
    [0.00, 0.00, 0.00, 0.00, 0.00],
])


def _soft_bond_order_components(pred_pos, pred_type_probs, config):
    d = torch.cdist(pred_pos, pred_pos)
    temp = config["bond_temperature"]
    p1 = _soft_pair_compatibility(pred_type_probs, BOND_ORDER_1_THRESHOLDS) * torch.sigmoid(
        (_soft_pair_threshold(pred_type_probs, BOND_ORDER_1_THRESHOLDS) - d) / temp
    )
    p2 = _soft_pair_compatibility(pred_type_probs, BOND_ORDER_2_THRESHOLDS) * torch.sigmoid(
        (_soft_pair_threshold(pred_type_probs, BOND_ORDER_2_THRESHOLDS) - d) / temp
    )
    p3 = _soft_pair_compatibility(pred_type_probs, BOND_ORDER_3_THRESHOLDS) * torch.sigmoid(
        (_soft_pair_threshold(pred_type_probs, BOND_ORDER_3_THRESHOLDS) - d) / temp
    )
    return p1, p2, p3


def _generic_valence_prior_losses(pred_pos, pred_type_probs, config):
    batch_size, n_atoms, _ = pred_pos.shape
    eye = torch.eye(n_atoms, device=pred_pos.device, dtype=torch.bool).unsqueeze(0)

    d = torch.cdist(pred_pos, pred_pos).masked_fill(eye, float("inf"))
    clash_loss = torch.relu(config["clash_threshold"] - d).pow(2).sum(dim=(1, 2)).mean() / (n_atoms * (n_atoms - 1))

    p1, p2, p3 = _soft_bond_order_components(pred_pos, pred_type_probs, config)
    p1 = p1.masked_fill(eye, 0.0)
    p2 = p2.masked_fill(eye, 0.0)
    p3 = p3.masked_fill(eye, 0.0)
    soft_order = p1 + p2 + p3
    soft_valence = soft_order.sum(dim=-1)

    typical_valence = ATOM_TYPICAL_VALENCE.to(pred_pos.device, dtype=pred_pos.dtype)
    expected_valence = pred_type_probs @ typical_valence
    valence_excess_loss = torch.relu(soft_valence - expected_valence).pow(2).mean()

    hydrogen_prob = pred_type_probs[..., 0]
    single_degree = p1.sum(dim=-1)
    second_bond_prob = p1.topk(k=2, dim=-1).values[..., 1]
    hydrogen_valence_loss = (
        hydrogen_prob * (single_degree - 1.0).pow(2)
        + 2.0 * hydrogen_prob * second_bond_prob.pow(2)
        + hydrogen_prob * (p2 + p3).sum(dim=-1).pow(2)
    ).mean()

    total_aux = (
        config["lambda_clash"] * clash_loss
        + config["lambda_valence_excess"] * valence_excess_loss
        + config["lambda_hydrogen_valence"] * hydrogen_valence_loss
    )

    return total_aux, {
        "clash_loss": clash_loss,
        "valence_excess_loss": valence_excess_loss,
        "hydrogen_valence_loss": hydrogen_valence_loss,
    }


def _auxiliary_generation_losses(pred_pos, pred_type_probs, real_pos, real_types, config):
    del real_pos, real_types
    return _generic_valence_prior_losses(pred_pos, pred_type_probs, config)



def _lr_factor_for_step(step, schedule):
    factor = 1.0
    for milestone, milestone_factor in schedule:
        if step >= milestone:
            factor = milestone_factor
    return factor


def _checkpoint_score(row):
    if "mol_stability" in row:
        return (
            1.5 * (1.0 - row["mol_stability"])
            + 0.5 * (1.0 - row["atom_stability"])
            + 0.2 * (1.0 - row["validity"])
            + 0.25 * row["diag_rmse"]
            + 0.1 * (1.0 - row["nearest_real_acc"])
        )
    return (
        row["diag_rmse"]
        + 0.5 * (1.0 - row["nearest_real_acc"])
        + 0.25 * (1.0 - row["type_acc"])
        + 0.05 * row["clash_loss"]
        + 0.05 * row["valence_excess_loss"]
        + 0.05 * row["hydrogen_valence_loss"]
    )


def _clone_model_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def _chemistry_training_metrics(pred_pos, pred_type_probs):
    try:
        from model.mol_utils import batch_to_stability, batch_to_validity
    except ModuleNotFoundError:
        return {}

    labels = pred_type_probs.argmax(dim=-1).detach().cpu()
    pos = pred_pos.detach().cpu()
    batch_size, n_atoms = labels.shape
    flat_pos = pos.reshape(batch_size * n_atoms, 3)
    flat_labels = labels.reshape(batch_size * n_atoms)
    batch_vec = torch.arange(batch_size).repeat_interleave(n_atoms)

    atom_stability, mol_stability = batch_to_stability(flat_pos, flat_labels, batch_vec)
    validity_results = batch_to_validity(flat_pos, flat_labels, batch_vec)
    validity = sum(ok for ok, _ in validity_results) / len(validity_results)
    return {
        "validity": float(validity),
        "atom_stability": float(atom_stability),
        "mol_stability": float(mol_stability),
    }


@torch.no_grad()
def _alignment_training_diagnostics(pred_pos, pred_types, real_pos, real_types, config):
    assignment, R, aligned_pos, aligned_types = find_rotation_and_permutation(
        pred_pos,
        real_pos,
        pred_types,
        real_types,
        sigma=config["posit_sigma"],
        eps=1e-8,
        max_iter=10,
        **config["alignment_kwargs"],
    )

    n = pred_pos.shape[0]
    arange = torch.arange(n, device=pred_pos.device)
    pair_rmse = (aligned_pos - real_pos[None, :, :, :]).pow(2).sum(dim=-1).mean(dim=-1).sqrt()
    diag_rmse = pair_rmse[arange, arange]
    nearest_real = pair_rmse.argmin(dim=1)
    diag_type_acc = (
        aligned_types[arange, arange].argmax(dim=-1) == real_types.argmax(dim=-1)
    ).float().mean(dim=-1)

    return {
        "diag_rmse": diag_rmse.mean().item(),
        "nearest_real_acc": (nearest_real == arange).float().mean().item(),
        "type_acc": diag_type_acc.mean().item(),
    }


def train_geometry_w1_drifting_model(config=DRIFT_TRAIN_CONFIG):
    torch.manual_seed(config["seed"])
    device = torch.device(config["device"])

    real_pos, real_types, selected_indices = _select_real_molecule_batch(
        size=config["size"],
        batch_size=config["batch_size"],
        max_scan=config["max_scan"],
        device=device,
    )
    batch_size, n_atoms, num_atom_types = real_types.shape
    edge_index = _make_complete_edge_index_fixed_batch(batch_size, n_atoms, device)

    model = EGNN(
        num_atom_types=num_atom_types,
        num_blocks=config["num_blocks"],
        hidden_nf=config["hidden_nf"],
        num_layers_per_block=config["num_layers_per_block"],
        coords_range=config["coords_range"],
    ).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    history = []
    best_checkpoint = None
    best_score = float("inf")
    print(
        f"Training geometry_w1 drift model on {batch_size} real molecules "
        f"with {n_atoms} atoms each. Dataset indices: {selected_indices[:5]}..."
    )

    for step in range(1, config["steps"] + 1):
        model.train()
        clean_pos = real_pos
        clean_types = probs_to_sphere(real_types)

        pred_pos, pred_type_logits = model(
            clean_pos.reshape(batch_size * n_atoms, 3),
            clean_types.reshape(batch_size * n_atoms, num_atom_types),
            edge_index,
        )
        pred_pos = pred_pos.reshape(batch_size, n_atoms, 3)
        pred_pos = pred_pos - pred_pos.mean(dim=1, keepdim=True)
        pred_type_probs = torch.softmax(
            pred_type_logits.reshape(batch_size, n_atoms, num_atom_types),
            dim=-1,
        )
        pred_types = probs_to_sphere(pred_type_probs)

        drift_loss, molecule_position_dist, molecule_types_dist = compute_aligning_drift_loss(
            pred_pos,
            real_pos,
            pred_types,
            real_types,
            posit_sigma=config["posit_sigma"],
            types_sigma=config["types_sigma"],
            scale_loss=config["scale_loss"],
            alignment_kwargs=config["alignment_kwargs"],
        )
        aux_loss, aux_parts = _auxiliary_generation_losses(
            pred_pos,
            pred_type_probs,
            real_pos,
            real_types,
            config,
        )
        loss = config["lambda_drift"] * drift_loss + aux_loss

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss at step {step}: {loss.item()}")

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
        optimizer.step()

        lr_factor = _lr_factor_for_step(step, config["lr_schedule"])
        for group in optimizer.param_groups:
            group["lr"] = config["lr"] * lr_factor

        row = {
            "step": step,
            "loss": loss.item(),
            "drift_loss": drift_loss.item(),
            "weighted_drift_loss": (config["lambda_drift"] * drift_loss).item(),
            "aux_loss": aux_loss.item(),
            "position_loss": molecule_position_dist.mean().item(),
            "type_loss": molecule_types_dist.mean().item(),
            "clash_loss": aux_parts["clash_loss"].item(),
            "valence_excess_loss": aux_parts["valence_excess_loss"].item(),
            "hydrogen_valence_loss": aux_parts["hydrogen_valence_loss"].item(),
            "grad_norm": float(grad_norm),
            "lr": optimizer.param_groups[0]["lr"],
        }

        should_log = step == 1 or step % config["log_every"] == 0
        should_chemistry_checkpoint = step == 1 or step % config["chemistry_checkpoint_every"] == 0
        should_diagnostics = should_log or should_chemistry_checkpoint or step % config["checkpoint_every"] == 0
        if should_diagnostics:
            diagnostics = _alignment_training_diagnostics(pred_pos, pred_types, real_pos, real_types, config)
            row.update(diagnostics)

            if should_chemistry_checkpoint:
                row.update(_chemistry_training_metrics(pred_pos, pred_type_probs))

            score = _checkpoint_score(row)
            row["checkpoint_score"] = score

            if should_chemistry_checkpoint and score < best_score:
                best_score = score
                best_checkpoint = {
                    "step": step,
                    "score": score,
                    "state_dict": _clone_model_state(model),
                    "row": dict(row),
                }

            chem_msg = ""
            if "mol_stability" in row:
                chem_msg = (
                    f" | valid={row['validity']:.3f} | atom_stab={row['atom_stability']:.3f} "
                    f"| mol_stab={row['mol_stability']:.3f}"
                )

            if should_log or should_chemistry_checkpoint:
                    print(
                    f"step {step:04d} | loss={row['loss']:.6f} | "
                    f"drift={row['drift_loss']:.6f} | w_drift={row['weighted_drift_loss']:.6f} | "
                    f"aux={row['aux_loss']:.6f} | "
                    f"pos={row['position_loss']:.6f} | type={row['type_loss']:.6f} | "
                    f"clash={row['clash_loss']:.4f} | "
                    f"valence_excess={row['valence_excess_loss']:.4f} | "
                    f"h_val={row['hydrogen_valence_loss']:.4f} | "
                    f"diag_rmse={row['diag_rmse']:.4f} | nearest={row['nearest_real_acc']:.3f} | "
                    f"type_acc={row['type_acc']:.3f} | lr={row['lr']:.1e} | "
                    f"score={row['checkpoint_score']:.4f} | grad={row['grad_norm']:.3f}"
                    f"{chem_msg}"
                )

        history.append(row)

    if best_checkpoint is not None:
        model.load_state_dict(best_checkpoint["state_dict"])
        best_row = best_checkpoint["row"]
        if config.get("checkpoint_path"):
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "history": history,
                    "best_checkpoint": {
                        "step": best_checkpoint["step"],
                        "score": best_checkpoint["score"],
                        "row": best_checkpoint["row"],
                    },
                    "real_pos": real_pos.detach().cpu(),
                    "real_types": real_types.detach().cpu(),
                    "selected_indices": selected_indices,
                    "config": dict(config),
                },
                config["checkpoint_path"],
            )
            print(f"Saved best checkpoint to {config['checkpoint_path']}")
        print(
            f"Loaded best checkpoint from step {best_checkpoint['step']} | "
            f"score={best_checkpoint['score']:.4f} | "
            f"diag_rmse={best_row['diag_rmse']:.4f} | "
            f"nearest={best_row['nearest_real_acc']:.3f} | "
            f"type_acc={best_row['type_acc']:.3f}"
            + (
                f" | valid={best_row['validity']:.3f} | atom_stab={best_row['atom_stability']:.3f} "
                f"| mol_stab={best_row['mol_stability']:.3f}"
                if "mol_stability" in best_row else ""
            )
        )

    return model, history, real_pos, real_types, selected_indices


geometry_w1_model, geometry_w1_history, drift_real_pos, drift_real_types, drift_selected_indices = train_geometry_w1_drifting_model()


Training geometry_w1 drift model on 10 real molecules with 6 atoms each. Dataset indices: [7, 9, 11, 22, 25]...
step 0001 | loss=0.107674 | drift=0.100586 | w_drift=0.100586 | aux=0.007088 | pos=0.023127 | type=0.077459 | clash=0.0000 | valence_excess=0.0177 | h_val=0.1347 | diag_rmse=0.4502 | nearest=0.600 | type_acc=0.133 | lr=1.0e-05 | score=2.0859 | grad=1.145 | valid=0.000 | atom_stab=0.533 | mol_stab=0.000
step 0010 | loss=0.110029 | drift=0.103541 | w_drift=0.103541 | aux=0.006488 | pos=0.029684 | type=0.073857 | clash=0.0000 | valence_excess=0.0126 | h_val=0.1247 | diag_rmse=0.5191 | nearest=0.400 | type_acc=0.150 | lr=1.0e-05 | score=1.0385 | grad=0.738
step 0020 | loss=0.099748 | drift=0.093571 | w_drift=0.093571 | aux=0.006177 | pos=0.022987 | type=0.070584 | clash=0.0000 | valence_excess=0.0095 | h_val=0.1197 | diag_rmse=0.6064 | nearest=0.300 | type_acc=0.283 | lr=1.0e-05 | score=1.1420 | grad=0.411
step 0030 | loss=0.101360 | drift=0.095061 | w_drift=0.095061 | aux=0.0062

In [32]:
def _resolve_geometry_w1_checkpoint_path(path=None):
    path = Path(path or DRIFT_TRAIN_CONFIG["checkpoint_path"])
    candidates = [
        path,
        Path.cwd() / path,
        Path.cwd() / "notebooks" / path.name,
        Path.cwd().parent / path.name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return path


def load_geometry_w1_checkpoint(path=None, device=None):
    path = _resolve_geometry_w1_checkpoint_path(path)
    device = torch.device(device or DRIFT_TRAIN_CONFIG["device"])
    ckpt = torch.load(path, map_location=device)

    real_pos = ckpt["real_pos"].to(device)
    real_types = ckpt["real_types"].to(device)
    config = dict(DRIFT_TRAIN_CONFIG)
    config.update(ckpt.get("config", {}))

    model = EGNN(
        num_atom_types=real_types.shape[-1],
        num_blocks=config["num_blocks"],
        hidden_nf=config["hidden_nf"],
        num_layers_per_block=config["num_layers_per_block"],
        coords_range=config["coords_range"],
    ).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    return (
        model,
        ckpt.get("history", []),
        real_pos,
        real_types,
        ckpt.get("selected_indices", []),
        ckpt,
    )


# Run this after restarting the kernel and rerunning the definition cells:
# geometry_w1_model, geometry_w1_history, drift_real_pos, drift_real_types, drift_selected_indices, geometry_w1_ckpt = load_geometry_w1_checkpoint()


In [12]:
# Drift-off ablation: chemistry/geometric auxiliary losses only.
# Expected outcome: if drifting is necessary, this should not preserve identity/alignment as well.
DRIFT_OFF_CONFIG = dict(DRIFT_TRAIN_CONFIG)
DRIFT_OFF_CONFIG.update({
    "lambda_drift": 0.0,
    "checkpoint_path": "geometry_w1_drift_off_checkpoint.pt",
})

drift_off_model, drift_off_history, drift_off_real_pos, drift_off_real_types, drift_off_selected_indices = train_geometry_w1_drifting_model(DRIFT_OFF_CONFIG)


Training geometry_w1 drift model on 2 real molecules with 3 atoms each. Dataset indices: [2, 4]...
step 0001 | loss=0.009735 | drift=0.145424 | w_drift=0.000000 | aux=0.009735 | pos=0.008270 | type=0.137154 | clash=0.0000 | valence_excess=0.0000 | h_val=0.1947 | diag_rmse=0.0001 | nearest=1.000 | type_acc=0.167 | lr=2.0e-04 | score=0.2182 | grad=0.181
step 0010 | loss=0.001369 | drift=0.199991 | w_drift=0.000000 | aux=0.001369 | pos=0.030877 | type=0.169115 | clash=0.0000 | valence_excess=0.0000 | h_val=0.0274 | diag_rmse=0.2933 | nearest=1.000 | type_acc=0.167 | lr=2.0e-04 | score=0.5030 | grad=0.019
step 0020 | loss=0.000105 | drift=0.263069 | w_drift=0.000000 | aux=0.000105 | pos=0.040683 | type=0.222386 | clash=0.0000 | valence_excess=0.0000 | h_val=0.0021 | diag_rmse=0.3579 | nearest=1.000 | type_acc=0.167 | lr=2.0e-04 | score=0.5664 | grad=0.004
step 0030 | loss=0.000003 | drift=0.275746 | w_drift=0.000000 | aux=0.000003 | pos=0.024181 | type=0.251565 | clash=0.0000 | valence_exc

KeyboardInterrupt: 

In [33]:
torch.save(
    {
        "model_state_dict": geometry_w1_model.state_dict(),
        "history": geometry_w1_history,
        "real_pos": drift_real_pos.detach().cpu(),
        "real_types": drift_real_types.detach().cpu(),
        "selected_indices": drift_selected_indices,
        "config": DRIFT_TRAIN_CONFIG,
    },
    "geometry_w1_best_checkpoint.pt",
)

# Generated molecule evaluation

This cell evaluates the trained `geometry_w1_model` on clean molecules using RDKit-backed project utilities from `model.mol_utils`.

In [34]:
from collections import Counter
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "model").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from model.mol_utils import batch_to_stability, batch_to_validity, heavy_atom_counts
from model.mol_utils.constants import _ATOM_NAMES

ATOM_SYMBOLS = list(_ATOM_NAMES)


def _predict_on_clean_real_batch(model, real_pos, real_types):
    model.eval()
    batch_size, n_atoms, num_atom_types = real_types.shape
    device = real_pos.device
    edge_index = _make_complete_edge_index_fixed_batch(batch_size, n_atoms, device)

    with torch.no_grad():
        pred_pos, pred_type_logits = model(
            real_pos.reshape(batch_size * n_atoms, 3),
            probs_to_sphere(real_types).reshape(batch_size * n_atoms, num_atom_types),
            edge_index,
        )
        pred_pos = pred_pos.reshape(batch_size, n_atoms, 3)
        pred_pos = pred_pos - pred_pos.mean(dim=1, keepdim=True)
        pred_type_probs = torch.softmax(
            pred_type_logits.reshape(batch_size, n_atoms, num_atom_types),
            dim=-1,
        )
        pred_types = probs_to_sphere(pred_type_probs)

    return pred_pos, pred_types, pred_type_probs


def _flatten_molecules(pos, atom_type_labels):
    batch_size, n_atoms, _ = pos.shape
    batch_vec = torch.arange(batch_size, device=pos.device).repeat_interleave(n_atoms)
    return (
        pos.reshape(batch_size * n_atoms, 3).detach().cpu(),
        atom_type_labels.reshape(batch_size * n_atoms).detach().cpu(),
        batch_vec.detach().cpu(),
    )


def _mol_utils_metrics(pos, atom_type_labels):
    flat_pos, flat_types, batch_vec = _flatten_molecules(pos, atom_type_labels)
    validity_results = batch_to_validity(flat_pos, flat_types, batch_vec)
    atom_stability, mol_stability = batch_to_stability(flat_pos, flat_types, batch_vec)
    heavy_counts = heavy_atom_counts(flat_types, batch_vec)

    valid_ids = [identifier for ok, identifier in validity_results if ok and identifier is not None]
    validity = sum(1 for ok, _ in validity_results if ok) / len(validity_results)
    uniqueness = len(set(valid_ids)) / len(valid_ids) if valid_ids else 0.0

    return {
        "validity": validity,
        "uniqueness": uniqueness,
        "atom_stability": atom_stability,
        "mol_stability": mol_stability,
        "heavy_atom_mean": sum(heavy_counts) / len(heavy_counts),
        "validity_results": validity_results,
        "valid_ids": valid_ids,
        "heavy_counts": heavy_counts,
    }


def _pairwise_distance_metrics(pos):
    d = torch.cdist(pos, pos)
    n = pos.shape[-2]
    eye = torch.eye(n, device=pos.device, dtype=torch.bool).unsqueeze(0)
    d_no_self = d.masked_fill(eye, float("inf"))
    min_dist = d_no_self.amin(dim=(1, 2))
    nn_dist = d_no_self.amin(dim=2).mean(dim=1)
    radius_gyration = pos.pow(2).sum(dim=-1).mean(dim=-1).sqrt()
    return min_dist, nn_dist, radius_gyration


def _type_count_l1(pred_probs, real_types):
    pred_counts = pred_probs.argmax(dim=-1)
    n_types = real_types.shape[-1]
    pred_one_hot = torch.nn.functional.one_hot(pred_counts, num_classes=n_types).float()
    return (pred_one_hot.sum(dim=1) - real_types.sum(dim=1)).abs().sum(dim=-1)


def _plot_generated_examples(pred_pos, pred_probs, real_pos, real_types, example_indices=(0, 1, 2, 3)):
    for idx in example_indices:
        if idx >= pred_pos.shape[0]:
            continue
        fig = plt.figure(figsize=(10, 4))
        labels = [pred_probs[idx].argmax(dim=-1), real_types[idx].argmax(dim=-1)]
        positions = [pred_pos[idx], real_pos[idx]]
        titles = ["Generated", "Real"]

        for col in range(2):
            ax = fig.add_subplot(1, 2, col + 1, projection="3d")
            pos_np = positions[col].detach().cpu().numpy()
            label_cpu = labels[col].detach().cpu()
            for atom_type in torch.unique(label_cpu):
                mask = (label_cpu == atom_type).numpy()
                symbol = ATOM_SYMBOLS[int(atom_type)] if int(atom_type) < len(ATOM_SYMBOLS) else str(int(atom_type))
                ax.scatter(pos_np[mask, 0], pos_np[mask, 1], pos_np[mask, 2], label=symbol)
            ax.set_title(f"{titles[col]} #{idx}")
            ax.legend()
        plt.tight_layout()
        plt.show()


def _ensure_geometry_w1_checkpoint_loaded():
    required = ["geometry_w1_model", "drift_real_pos", "drift_real_types"]
    missing = [name for name in required if name not in globals()]
    if not missing:
        return

    if "load_geometry_w1_checkpoint" not in globals():
        raise RuntimeError(
            "Missing trained model globals and load_geometry_w1_checkpoint is not defined. "
            "After restarting, rerun the notebook definition cells through the checkpoint reload cell first."
        )

    checkpoint_path = "/Users/edovergna/Desktop/DL2/drifting-experiments/notebooks/geometry_w1_drift_off_checkpoint.pt"
    #DRIFT_TRAIN_CONFIG.get("checkpoint_path", "geometry_w1_drift_off_checkpoint.pt")
    #if "_resolve_geometry_w1_checkpoint_path" in globals():
    #    checkpoint_path = _resolve_geometry_w1_checkpoint_path(checkpoint_path)
    #else:
    #    raw_path = Path(checkpoint_path)
    #    candidates = [raw_path, Path.cwd() / raw_path, Path.cwd() / "notebooks" / raw_path.name, Path.cwd().parent / raw_path.name]
    #   checkpoint_path = next((candidate for candidate in candidates if candidate.exists()), raw_path)

    if not Path(checkpoint_path).exists():
        raise RuntimeError(
            f"Missing trained model globals and checkpoint file was not found at {Path(checkpoint_path).resolve()}. "
            "Run training or save the checkpoint before evaluation."
        )

    (
        globals()["geometry_w1_model"],
        globals()["geometry_w1_history"],
        globals()["drift_real_pos"],
        globals()["drift_real_types"],
        globals()["drift_selected_indices"],
        globals()["geometry_w1_ckpt"],
    ) = load_geometry_w1_checkpoint(checkpoint_path)
    print(f"Loaded checkpoint from {Path(checkpoint_path).resolve()}")


def evaluate_generated_molecules(num_examples=4):
    _ensure_geometry_w1_checkpoint_loaded()

    pred_pos, pred_types, pred_probs = _predict_on_clean_real_batch(
        geometry_w1_model,
        drift_real_pos,
        drift_real_types,
    )
    pred_labels = pred_probs.argmax(dim=-1)
    real_labels = drift_real_types.argmax(dim=-1)

    assignment, R, aligned_pos, aligned_types = find_rotation_and_permutation(
        pred_pos,
        drift_real_pos,
        pred_types,
        drift_real_types,
        sigma=DRIFT_TRAIN_CONFIG["posit_sigma"],
        eps=1e-8,
        max_iter=10,
        **DRIFT_TRAIN_CONFIG["alignment_kwargs"],
    )

    n = pred_pos.shape[0]
    arange = torch.arange(n, device=pred_pos.device)
    pair_rmse = (aligned_pos - drift_real_pos[None, :, :, :]).pow(2).sum(dim=-1).mean(dim=-1).sqrt()
    diag_rmse = pair_rmse[arange, arange]
    nearest_real = pair_rmse.argmin(dim=1)
    type_acc = (
        aligned_types[arange, arange].argmax(dim=-1) == real_labels
    ).float().mean(dim=-1)

    gen_min_dist, gen_nn_dist, gen_rg = _pairwise_distance_metrics(pred_pos)
    real_min_dist, real_nn_dist, real_rg = _pairwise_distance_metrics(drift_real_pos)
    type_l1 = _type_count_l1(pred_probs, drift_real_types)

    gen_chem = _mol_utils_metrics(pred_pos, pred_labels)
    real_chem = _mol_utils_metrics(drift_real_pos, real_labels)

    print("Generated molecule aggregate metrics")
    print(f"aligned diag RMSE: mean={diag_rmse.mean().item():.4f}, median={diag_rmse.median().item():.4f}, max={diag_rmse.max().item():.4f}")
    print(f"nearest-real acc: {(nearest_real == arange).float().mean().item():.4f}")
    print(f"aligned type acc: {type_acc.mean().item():.4f}")
    print(f"atom-count L1 error: mean={type_l1.mean().item():.4f}, max={type_l1.max().item():.4f}")
    print(f"min pairwise distance: generated mean={gen_min_dist.mean().item():.4f}, real mean={real_min_dist.mean().item():.4f}")
    print(f"nearest-neighbor distance: generated mean={gen_nn_dist.mean().item():.4f}, real mean={real_nn_dist.mean().item():.4f}")
    print(f"radius of gyration: generated mean={gen_rg.mean().item():.4f}, real mean={real_rg.mean().item():.4f}")

    print("\nRDKit/mol_utils metrics")
    print(
        f"generated: validity={gen_chem['validity']:.4f}, uniqueness={gen_chem['uniqueness']:.4f}, "
        f"atom_stability={gen_chem['atom_stability']:.4f}, mol_stability={gen_chem['mol_stability']:.4f}, "
        f"heavy_atom_mean={gen_chem['heavy_atom_mean']:.2f}"
    )
    print(
        f"real:      validity={real_chem['validity']:.4f}, uniqueness={real_chem['uniqueness']:.4f}, "
        f"atom_stability={real_chem['atom_stability']:.4f}, mol_stability={real_chem['mol_stability']:.4f}, "
        f"heavy_atom_mean={real_chem['heavy_atom_mean']:.2f}"
    )

    if gen_chem["valid_ids"]:
        print("\nMost common generated valid SMILES")
        for smiles, count in Counter(gen_chem["valid_ids"]).most_common(10):
            print(f"{count:>3}  {smiles}")
    else:
        print("\nNo generated valid SMILES found.")

    print("\nPer-molecule sample")
    for idx in range(min(num_examples, n)):
        pred_symbols = [ATOM_SYMBOLS[int(x)] if int(x) < len(ATOM_SYMBOLS) else str(int(x)) for x in pred_labels[idx].detach().cpu()]
        real_symbols = [ATOM_SYMBOLS[int(x)] if int(x) < len(ATOM_SYMBOLS) else str(int(x)) for x in real_labels[idx].detach().cpu()]
        valid_ok, identifier = gen_chem["validity_results"][idx]
        print(
            f"#{idx:02d} | diag_rmse={diag_rmse[idx].item():.4f} | nearest={int(nearest_real[idx].item())} | "
            f"type_acc={type_acc[idx].item():.3f} | type_count_l1={type_l1[idx].item():.0f} | "
            f"min_dist={gen_min_dist[idx].item():.3f} | valid={valid_ok} | id={identifier} | "
            f"pred={pred_symbols} | real={real_symbols}"
        )

    _plot_generated_examples(
        pred_pos,
        pred_probs,
        drift_real_pos,
        drift_real_types,
        example_indices=tuple(range(min(num_examples, n))),
    )

    return {
        "pred_pos": pred_pos,
        "pred_type_probs": pred_probs,
        "aligned_pos": aligned_pos,
        "aligned_types": aligned_types,
        "diag_rmse": diag_rmse,
        "nearest_real": nearest_real,
        "type_acc": type_acc,
        "type_count_l1": type_l1,
        "generated_chem": gen_chem,
        "real_chem": real_chem,
    }


generated_eval = evaluate_generated_molecules(num_examples=100)


Generated molecule aggregate metrics
aligned diag RMSE: mean=0.7208, median=0.5451, max=1.4795
nearest-real acc: 0.7000
aligned type acc: 0.7833
atom-count L1 error: mean=1.4000, max=4.0000
min pairwise distance: generated mean=0.8313, real mean=1.0768
nearest-neighbor distance: generated mean=1.1665, real mean=1.1397
radius of gyration: generated mean=1.7549, real mean=1.5612

RDKit/mol_utils metrics
generated: validity=0.7000, uniqueness=1.0000, atom_stability=0.4500, mol_stability=0.1000, heavy_atom_mean=4.10
real:      validity=0.8000, uniqueness=1.0000, atom_stability=0.9000, mol_stability=0.8000, heavy_atom_mean=4.20

Most common generated valid SMILES
  1  [H]CC([H])([H])[H]
  1  [H]C([H])([H])C
  1  [H]C
  1  [H]C#CC([H])=O
  1  [H]CO
  1  NO
  1  C

Per-molecule sample
#00 | diag_rmse=0.5451 | nearest=0 | type_acc=0.833 | type_count_l1=2 | min_dist=0.666 | valid=True | id=[H]CC([H])([H])[H] | pred=['C', 'C', 'H', 'H', 'H', 'H'] | real=['C', 'O', 'H', 'H', 'H', 'H']
#01 | diag_

/var/folders/1y/0vbb_3gj6bv7dm7t81psd_l40000gp/T/ipykernel_2718/2397643192.py:110: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/1y/0vbb_3gj6bv7dm7t81psd_l40000gp/T/ipykernel_2718/2397643192.py:94: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure(figsize=(10, 4))


In [16]:
def real_qm9_reconstructed_ids(size=8, max_scan=50000):
    ids = set()

    for idx in range(min(max_scan, len(dataset))):
        mol = dataset[idx]
        if int(mol.num_nodes) != size:
            continue

        pos = mol.pos.float()
        pos = pos - pos.mean(dim=0, keepdim=True)
        labels = mol.real_atom_types.argmax(dim=-1).long()

        batch_vec = torch.zeros(labels.shape[0], dtype=torch.long)
        result = batch_to_validity(pos.detach().cpu(), labels.detach().cpu(), batch_vec)[0]

        ok, identifier = result
        if ok and identifier is not None:
            ids.add(identifier)

    return ids


real_size8_ids = real_qm9_reconstructed_ids(size=8, max_scan=50000)
gen_ids = generated_eval["generated_chem"]["valid_ids"]

novel_vs_size8 = [
    smiles for smiles in gen_ids
    if smiles not in real_size8_ids
]

print(f"real valid unique size-8 reconstructed: {len(real_size8_ids)}")
print(f"generated valid molecules: {len(gen_ids)}")
print(f"generated novel vs scanned size-8 QM9: {len(novel_vs_size8)}")
print(f"novel fraction: {len(novel_vs_size8) / max(len(gen_ids), 1):.4f}")

novel_vs_size8[:20]

real valid unique size-8 reconstructed: 55
generated valid molecules: 64
generated novel vs scanned size-8 QM9: 64
novel fraction: 1.0000


['[HH]',
 'O',
 'O',
 '[HH]',
 '[HH]',
 '[HH]',
 '[HH]',
 '[HH]',
 'O',
 'O',
 '[HH]',
 '[HH]',
 'O',
 '[HH]',
 '[HH]',
 'O',
 'OOO',
 'OO',
 'OO',
 'OOO']

In [37]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ATOM_COLORS = {
    "H": "#d9d9d9",
    "C": "#222222",
    "N": "#2f6fff",
    "O": "#e53935",
    "F": "#43a047",
}

def plot_generated_vs_real_3d(eval_result=generated_eval, idx=0):
    pred_pos = eval_result["pred_pos"].detach().cpu()
    pred_labels = eval_result["pred_type_probs"].argmax(dim=-1).detach().cpu()
    real_pos = drift_real_pos.detach().cpu()
    real_labels = drift_real_types.argmax(dim=-1).detach().cpu()

    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "scene"}, {"type": "scene"}]],
        subplot_titles=("Generated", "Real"),
    )

    for col, pos, labels, name in [
        (1, pred_pos[idx], pred_labels[idx], "generated"),
        (2, real_pos[idx], real_labels[idx], "real"),
    ]:
        for atom_idx, atom_name in enumerate(ATOM_SYMBOLS):
            mask = labels == atom_idx
            if not mask.any():
                continue

            xyz = pos[mask]
            fig.add_trace(
                go.Scatter3d(
                    x=xyz[:, 0],
                    y=xyz[:, 1],
                    z=xyz[:, 2],
                    mode="markers",
                    marker=dict(
                        size=7 if atom_name != "H" else 4,
                        color=ATOM_COLORS.get(atom_name, "#999999"),
                        line=dict(width=1, color="#111111"),
                    ),
                    name=f"{name} {atom_name}",
                    text=[atom_name] * xyz.shape[0],
                    hovertemplate="%{text}<br>x=%{x:.2f}<br>y=%{y:.2f}<br>z=%{z:.2f}<extra></extra>",
                    showlegend=(col == 1),
                ),
                row=1,
                col=col,
            )

    fig.update_layout(
        title=f"Molecule #{idx}",
        width=950,
        height=450,
        margin=dict(l=0, r=0, t=50, b=0),
    )
    fig.update_scenes(aspectmode="data")
    fig.show()

plot_generated_vs_real_3d(idx=4)